# QICK Job Server — Single Qubit Calibration

透過 Job Server 提交量測到佇列，由 Worker 依序執行。

**啟動前確認**:
1. Server 已啟動: `python -m uvicorn qick_workspace.qick_job_server.server:app --port 8585`
2. Worker 已啟動: `python -m qick_workspace.qick_job_server.worker`

# Load QICK & Client

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from qick import *
from qick.pyro import make_proxy
from qick import QickConfig
from qick.asm_v2 import QickSpan, QickSweep1D
import Pyro4

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

ns_host = "192.168.10.82"
ns_port = 8888
proxy_name = "myqick"

soc, soccfg = make_proxy(ns_host=ns_host, ns_port=ns_port, proxy_name=proxy_name)
print(soccfg)

c:\Users\cluster\anaconda3\envs\qick2env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
QICK library version mismatch: 0.2.369 remote (the board), 0.2.381 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_e7caa3fd81664172a7002125752b0f5b@192.168.10.82:32817
QICK running on ZCU216, software version 0.2.369

Firmware configuration (built Sun Sep  8 18:58:57 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 3], [DAC tile 2], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 0 is 0_230 on JHC3, or QICK box DAC port 8
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 4096 complex samples (0.427 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 1 is 1_230 on JHC4, or QICK box DAC port 9
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 8192 complex samples (0.855 us)
		32-bit DDS, range=9584.

In [2]:
from qick_workspace.qick_job_server.client import JobClient

client = JobClient("http://127.0.0.1:8585")
client.health_check()

{'status': 'healthy',
 'database_connected': True,
 'pending_jobs': 0,
 'running_jobs': 1}

# Qubit Setting

In [5]:
from qick_workspace.tools.system_tool import ExperimentConfig
from qick_workspace.tools.system_cfg import config_list

qubit = "Q1"
config_all = ExperimentConfig(config_list)
run_cfg = config_all.get_qubit(qubit)

In [6]:
print(config_all.to_yaml(q_id=1))

name: Q2

ch:
  res_ch: 0
  qb_ch: 7
  ro_ch: 0

res:
  res_freq_ge: 5351.559
  res_length: 10
  res_gain_ge: 0.1
  res_phase: 0
  res_sigma: 0.005
  ro_length: 5
  nqz_res: 2
  chi: false

qb:
  pulse_type: arb
  qb_freq_ge: 4000
  qb_mixer: 4000
  qb_gain_ge: 0.1
  qb_phase: 0
  sigma_ge: 0.05
  pi_gain_ge: 0.1
  pi2_gain_ge: 0.1
  qb_flat_top_length_ge: 0.1
  ramsey_freq: 2
  nqz_qb: 1

cooling:
  cooling: false
  cool_ch1: 2
  cool_freq_1: 5400
  cool_gain_1: 0.1
  nqz_cool_ch1: 2
  cool_length: 100
  cool_ch2: 3
  cool_freq_2: 5400
  cool_gain_2: 0.1
  nqz_cool_ch2: 2

reps: 100
trig_time: 0.5
relax_delay: 10


# Queue Status

In [3]:
client.print_queue()


=== QICK Job Queue ===

Running: JOB-20260302-00016
  User: jay
  Experiment: ResonatorSpec
  Qubit: Q1
  Started: 2026-03-02T05:37:26.221065

Pending: 0 jobs



# GE State

## Resonator OneTone

In [6]:
from qick_workspace.newscrip.s002_res_spec_ge import ResonatorSpec

START_FREQ = config_all.get_qubit(qubit)["res_freq_ge"] - 50  # [MHz]
STOP_FREQ = config_all.get_qubit(qubit)["res_freq_ge"] + 50  # [MHz]
STEPS = 101

config_all.update("res.res_gain_ge", 0.1, q_index=qubit)
run_cfg = config_all.get_qubit(qubit)

run_cfg.update(
    [
        ("steps", STEPS),
        ("res_freq_ge", QickSweep1D("freqloop", START_FREQ, STOP_FREQ)),
        ("relax_delay", 0),
    ]
)

onetone = ResonatorSpec(soc, soccfg, run_cfg)
onetone.subjob(py_avg=10, qubit=qubit, wait=True)
## update value ##
# config_all.update("res.res_freq_ge", round(fres[0] / 1e6, 4), q_index=qubit)
# onetone.saveLabber(qubit)

Job submitted: JOB-20260302-00017 (queue position: 1)

[0.0s] Job JOB-20260302-00017: pending

[66.4s] Job JOB-20260302-00017: running

[68.5s] Job JOB-20260302-00017: completed
[WORKER] Loading ResonatorSpec from qick_workspace.newscrip.s002_res_spec_ge
[WORKER] Creating experiment instance
[WORKER] Running experiment (simulate=False)...
Figure(600x400)
Software Average Count: 100%|██████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  8.41it/s]Figure(600x400)

{'Fres(GHz)': np.float64(5.3562),
 'Qi': -17,
 'Ql': 22,
 'absQc': 10,
 'κ(MHz)': np.float64(239.48)}
[WORKER] Saving expt object to: C:\Users\cluster\Desktop\SQC_soc-jobserver\qick_workspace\data\job_results\JOB-20260302-00017_ResonatorSpec.pkl

Job completed! Data: C:\Users\cluster\Desktop\SQC_soc-jobserver\qick_workspace\data\job_results\JOB-20260302-00017_ResonatorSpec.pkl
[ResonatorSpec] Job JOB-20260302-00017 complete. Data loaded.


array([  1.51708268+2.22198711e+01j, -19.00258464-1.17374479e+01j,
        20.62363346-8.62010677e+00j,  -4.98618294+2.17103932e+01j,
       -14.74217057-1.66912135e+01j,  22.10231315-2.26027995e+00j,
       -11.03305664+1.92822617e+01j,  -9.28495378-2.02854115e+01j,
        21.92765104+4.27211719e+00j, -16.22778906+1.53232767e+01j,
        -2.99803385-2.20469264e+01j,  19.63153711+1.03803867e+01j,
       -19.98794596+9.97397396e+00j,   3.53105794-2.20706732e+01j,
        15.87959635+1.57140469e+01j, -22.00775846+3.75916146e+00j,
         9.75101237-2.00759518e+01j,  10.67082943+1.96519967e+01j,
       -22.23488477-2.76838086e+00j,  15.23325977-1.65212389e+01j,
         4.5673138 +2.20027715e+01j, -20.50404102-9.06243685e+00j,
        19.30052409-1.14049069e+01j,  -1.95947005+2.24292871e+01j,
       -17.12213802-1.46524824e+01j,  21.90155859-5.41499284e+00j,
        -8.33421159+2.09869557e+01j, -12.21358203-1.89634824e+01j,
        22.57564844+1.08571745e+00j, -14.03122201+1.77758151e+

## Qubit Spectrum ge

In [ ]:
from qick_workspace.newscrip.s003_qubit_spec_ge import QubitSpec

center = 4500
SPAN = 200
START_FREQ = center - SPAN  # [MHz]
STOP_FREQ = center + SPAN  # [MHz]
STEPS = 201

config_all.update("ch.qb_ch", 2, q_index=qubit)
run_cfg = config_all.get_qubit(qubit)
run_cfg.update(
    [
        ("steps", STEPS),
        ("qb_freq_ge", QickSweep1D("freqloop", START_FREQ, STOP_FREQ)),
        ("qmixer_freq", center),
        ("qbgain_ge", 0.5),
        ("qb_length_ge", 5),
        ("relax_delay", 1),
        ("nqz_qb", 2),
    ]
)

spectrum_ge = QubitSpec(soc, soccfg, run_cfg)
spectrum_ge.subjob(py_avg=20, qubit=qubit)
# f_ge = spectrum_ge.run(20)
# config_all.update("qb.qb_freq_ge", f_ge, q_index=qubit)
# config_all.update("qb.qb_mixer", f_ge, q_index=qubit)

## Power Rabi ge

In [ ]:
from qick_workspace.newscrip.s005_power_rabi_ge import PowerRabi

START_GAIN = 0.0  # [DAC units]
STOP_GAIN = 1  # [DAC units]
STEPS = 100

config_all.update("qb.sigma", 0.02, q_index=qubit)
config_all.update("qb.nqz_qb", 1, q_index=qubit)
config_all.update("relax_delay", 30, q_index=qubit)
run_cfg = config_all.get_qubit(qubit)

run_cfg.update(
    [
        ("steps", STEPS),
        ("qb_gain_ge", QickSweep1D("gainloop", START_GAIN, STOP_GAIN)),
        ("cooling", False),
    ]
)

prabi = PowerRabi(soc, soccfg, run_cfg)
prabi.subjob(py_avg=20, qubit=qubit)
# pi_gain, pi2_gain = prabi.run(20)
# config_all.update("qb.pi_gain_ge", pi_gain, q_index=qubit)
# config_all.update("qb.pi2_gain_ge", pi2_gain, q_index=qubit)

## Ramsey ge

In [ ]:
from qick_workspace.newscrip.s006_Ramsey_ge import Ramsey

run_cfg = config_all.get_qubit(qubit)

START_TIME = 0.0  # [us]
STOP_TIME = 5  # [us]
STEPS = 100
run_cfg.update(
    [
        ("steps", STEPS),
        ("wait_time", QickSweep1D("waitloop", START_TIME, STOP_TIME)),
        ("ramsey_freq", 2),
    ]
)

t2r = Ramsey(soc, soccfg, run_cfg)
t2r.subjob(py_avg=10, qubit=qubit)
# t2r.saveLabber(qubit)

## T1 ge

In [ ]:
from qick_workspace.newscrip.s008_T1_ge import T1

run_cfg = config_all.get_qubit(qubit)

START_TIME = 0.0  # [us]
STOP_TIME = 50  # [us]
STEPS = 100
run_cfg.update(
    [
        ("steps", STEPS),
        ("wait_time", QickSweep1D("waitloop", START_TIME, STOP_TIME)),
        ("relax_delay", 50),
        ("cooling", False),
    ]
)

t1 = T1(soc, soccfg, run_cfg)
t1.subjob(py_avg=50, qubit=qubit)
# t1.saveLabber(qubit)

# Batch Submit (多筆一次提交)

In [ ]:
# 非同步提交多筆 job，不用等每筆完成再提交下一筆
from qick_workspace.newscrip.s006_Ramsey_ge import Ramsey
from qick_workspace.newscrip.s008_T1_ge import T1

# --- Ramsey ---
run_cfg_ramsey = config_all.get_qubit(qubit)
run_cfg_ramsey.update([
    ("steps", 100),
    ("wait_time", QickSweep1D("waitloop", 0.0, 5)),
    ("ramsey_freq", 2),
])
t2r = Ramsey(soc, soccfg, run_cfg_ramsey)
handle_ramsey = t2r.subjob(py_avg=10, qubit=qubit, wait=False)

# --- T1 ---
run_cfg_t1 = config_all.get_qubit(qubit)
run_cfg_t1.update([
    ("steps", 100),
    ("wait_time", QickSweep1D("waitloop", 0.0, 50)),
    ("relax_delay", 50),
    ("cooling", False),
])
t1_expt = T1(soc, soccfg, run_cfg_t1)
handle_t1 = t1_expt.subjob(py_avg=50, qubit=qubit, wait=False)

print("\nAll jobs submitted! Worker will execute them in order.")
client.print_queue()

In [ ]:
# 等待所有 job 完成
handle_ramsey.wait()
handle_t1.wait()

print("\nAll jobs done!")
print("Ramsey iqdata shape:", t2r.iqdata.shape if t2r.iqdata is not None else "N/A")
print("T1 iqdata shape:", t1_expt.iqdata.shape if t1_expt.iqdata is not None else "N/A")

# Job History

In [ ]:
import pandas as pd

history = client.get_history(limit=20)
pd.DataFrame(history)